# E8 · Candidatos de envío que cambian más filas
**Parte de:** E6 (stacking con *seed bagging* de 5 semillas; público 0,79100, empatado con la base 0,79160).

**Motivo.** E6 cambió solo 19 de 1.154 filas, así que el leaderboard público no puede distinguirlo de la base. Para aprovechar los envíos que quedan se generan dos variantes que cambian **más** predicciones, cada una por una razón concreta:

| Candidato | Razón |
|---|---|
| **E8a · E6 con umbral 0,45** | El modelo predice HS en el 39 % del test, frente al 44 % del train. Bajar el umbral da más positivos y mide si el test tiene más odio del que el modelo detecta |
| **E8b · bagging con los 5 modelos + LightGBM** | LightGBM es el único modelo no lineal y el menos correlacionado (E3). Combinarlo con el bagging da el candidato más "distinto" |

**Advertencia.** En la CV ninguno de los dos supera a E6 con claridad. Si alguno sube en el público puede ser ruido, así que la selección final se decide con la CV.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **Umbral** | Probabilidad a partir de la cual se predice 1 |
| **Seed bagging** | Promediar modelos entrenados con distintas particiones aleatorias |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


In [2]:
S5 = (42, 43, 44, 45, 46)
B5 = tb.BASES_ORIGINALES
B6 = B5 + ['LGBM']
for s in S5:
    for b in B6:
        tb.oof(d, b, s); tb.pred_test_folds(d, b, s)
    tb.log('semilla lista:', s)
E6 = {s: tb.stacking_bagged(d, B5, S5, s, con_test=False)[0] for s in tb.SEEDS}
R = {'E6 · bagging 5 semillas, umbral 0,5': tb.evaluar(d, E6),
     'E8a · E6 con umbral 0,45': tb.evaluar(d, E6, (0.45, 0.45, 0.45)),
     'E8b · bagging 5 modelos + LGBM': tb.evaluar(d, {s: tb.stacking_bagged(d, B6, S5, s, con_test=False)[0] for s in tb.SEEDS})}
tb.tabla(R)

[    23s] semilla lista: 42


[    44s] semilla lista: 43


[    64s] semilla lista: 44


[   104s] semilla lista: 45


[   141s] semilla lista: 46


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
"E6 · bagging 5 semillas, umbral 0,5",0.8092,0.8368,0.7988,0.8150,0.0007,0.7803,0.814 / 0.815 / 0.815
"E8a · E6 con umbral 0,45",0.8108,0.8387,0.8023,0.8173,0.0001,0.7789,0.817 / 0.817 / 0.817
E8b · bagging 5 modelos + LGBM,0.8071,0.8405,0.8062,0.8179,0.0012,0.7785,0.819 / 0.818 / 0.816


## Submissions

In [3]:
_, Pt6 = tb.stacking_bagged(d, B5, S5, 42)
_, Pt8b = tb.stacking_bagged(d, B6, S5, 42)
e6 = tb.binarizar(Pt6)
for nombre, Yt in [('toxic_submission_E8a_umbral045.csv', tb.binarizar(Pt6, (0.45, 0.45, 0.45))),
                   ('toxic_submission_E8b_bagging_lgbm.csv', tb.binarizar(Pt8b))]:
    print('=' * 20, nombre)
    sub = tb.guardar_submission(d, Yt, nombre)
    print('Filas distintas frente a E6:', int((sub[tb.LABELS].values != e6 * ~d.err_excel_te[:, None]).any(1).sum()))

==================== toxic_submission_E8a_umbral045.csv
[   142s] Guardada /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/experimentos/submissions/toxic_submission_E8a_umbral045.csv


Prevalencia train : {'HS': np.float64(0.437), 'TR': np.float64(0.3), 'AG': np.float64(0.382)}
Prevalencia pred. : {'HS': np.float64(0.407), 'TR': np.float64(0.308), 'AG': np.float64(0.38)}
Cambios frente a la base 0,79160: 28 filas ({'HS': 16, 'TR': 12, 'AG': 12})
Filas distintas frente a E6: 31
==================== toxic_submission_E8b_bagging_lgbm.csv
[   142s] Guardada /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/experimentos/submissions/toxic_submission_E8b_bagging_lgbm.csv


Prevalencia train : {'HS': np.float64(0.437), 'TR': np.float64(0.3), 'AG': np.float64(0.382)}
Prevalencia pred. : {'HS': np.float64(0.403), 'TR': np.float64(0.299), 'AG': np.float64(0.369)}
Cambios frente a la base 0,79160: 49 filas ({'HS': 37, 'TR': 15, 'AG': 17})
Filas distintas frente a E6: 44


## Conclusión
- Los dos candidatos superan levemente a E6 en la CV: **E8a 0,8173** y **E8b 0,8179**, frente a 0,8150. La mejora aparece en las 3 semillas del metamodelo, pero es de +0,002-0,003, en el límite del ruido.
- Cambian 31 y 44 filas respecto a E6, de 2 a 3 veces más que E6 frente a la base. Por eso el público puede reflejar algo de la diferencia, aunque con ~577 filas sigue siendo ruidoso.
- Archivos: `submissions/toxic_submission_E8a_umbral045.csv` y `submissions/toxic_submission_E8b_bagging_lgbm.csv`.